In [0]:
customers = spark.read.table("02_silver_catalog.transformed_tables.customers")
exchange_rates= spark.read.table("02_silver_catalog.transformed_tables.exchange_rates")
invoice_line_items = spark.read.table("02_silver_catalog.transformed_tables.invoice_line_items")
invoices = spark.read.table("02_silver_catalog.transformed_tables.invoices")
products = spark.read.table("02_silver_catalog.transformed_tables.products")
payments = spark.read.table("02_silver_catalog.transformed_tables.payments")
regions = spark.read.table("02_silver_catalog.transformed_tables.regions")
# display(customers)
# display(exchange_rates)
# display(invoice_line_items)
# display(invoices)
# display(products)
# display(payments)
# display(regions)

In [0]:
tables = [
    "customers",
    "exchange_rates",
    "invoice_line_items",
    "invoices",
    "products",
    "payments"
]

for table in tables:
    df = spark.read.table(f"02_silver_catalog.transformed_tables.{table}")
    df_clean = df.dropDuplicates()
    df_clean.write.mode("overwrite").saveAsTable(f"02_silver_catalog.transformed_tables.{table}")

In [0]:
tables = [
    "customers",
    "exchange_rates",
    "invoice_line_items",
    "invoices",
    "products",
    "payments"
]

for table in tables:
    df = spark.read.table(f"02_silver_catalog.transformed_tables.{table}")
    for col_name, dtype in df.dtypes:
        if dtype.startswith("string"):
            df = df.fillna({col_name: "unknown"})
        elif dtype in ["int", "bigint", "double", "float", "decimal"]:
            df = df.fillna({col_name: 0})
    df_clean = df.dropDuplicates()
    df_clean.write.mode("overwrite").saveAsTable(f"02_silver_catalog.transformed_tables.{table}")

In [0]:
from pyspark.sql.functions import col, to_date

exchange_rates = spark.read.table("02_silver_catalog.transformed_tables.exchange_rates") \
    .withColumn("date", to_date(col("date"), "yyyy/M/d"))
exchange_rates.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("02_silver_catalog.transformed_tables.exchange_rates")

invoices = spark.read.table("02_silver_catalog.transformed_tables.invoices") \
    .withColumn("invoice_date", to_date(col("invoice_date"), "yyyy/M/d"))
invoices.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("02_silver_catalog.transformed_tables.invoices")

payments = spark.read.table("02_silver_catalog.transformed_tables.payments") \
    .withColumn("payment_date", to_date(col("payment_date"), "yyyy/M/d"))
payments.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("02_silver_catalog.transformed_tables.payments")


In [0]:
%sql
CREATE OR REPLACE TABLE 02_silver_catalog.transformed_tables.fact_sales AS
SELECT
    ili.invoice_line_id,
    ili.invoice_id,
    ili.product_id,
    i.customer_id,
    i.invoice_date,
    i.region,
    i.currency,
    p.category,
    p.cost_price,
    quantity,
    unit_price,
    ROUND(quantity * unit_price, 2) AS gross_amount,
    ROUND((quantity * unit_price) * discount, 2) AS discount_amount,
    ROUND((quantity * unit_price) * (1 - discount), 2) AS net_amount_local,
    ROUND(er.rate_to_usd, 2) AS rate_to_usd,
    ROUND((quantity * unit_price) * (1 - discount) * er.rate_to_usd, 2) AS revenue_usd,
    ROUND((quantity * p.cost_price), 2) AS cost_total,
    ROUND(((quantity * unit_price) * (1 - discount)) - (quantity * p.cost_price), 2) AS profit_local,
    ROUND((((quantity * unit_price) * (1 - discount)) - (quantity * p.cost_price)) * er.rate_to_usd, 2) AS profit_usd
FROM 02_silver_catalog.transformed_tables.invoice_line_items ili
JOIN 02_silver_catalog.transformed_tables.products p
    ON ili.product_id = p.product_id
JOIN 02_silver_catalog.transformed_tables.invoices i
    ON ili.invoice_id = i.invoice_id
JOIN 02_silver_catalog.transformed_tables.exchange_rates er
    ON i.currency = er.currency
   AND i.invoice_date = er.date;